# 04 | Final Plots for the Paper

This notebook gathers the last figures and the reference values used to check the write-up.

It does three things:

1. ensures that the final paper figures exist
2. displays those figures in the same order as the paper
3. collects the reference values used in the text


## Setup

The earlier notebooks produced the study outputs.
Here we add the final degree-distribution fit if needed and then assemble the paper-ready material.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "rec_dating_project").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


project_root = locate_project_root()
sys.path.insert(0, str(project_root / "src"))

from rec_dating_project import PopularityPrestigeAnalyzer, ProjectPaths


paths = ProjectPaths.default()
paths.ensure_output_dirs()
OUTPUT_DATA = paths.output_data_dir
OUTPUT_FIGURES = paths.output_figures_dir
FORCE_REBUILD = False


def run_script(script_name: str, *args: str) -> None:
    cmd = [sys.executable, str(paths.scripts_dir / script_name), *args]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=paths.project_root)


def ensure_all_outputs(force: bool = False) -> None:
    stage_two = [
        OUTPUT_DATA / "profile_metrics_full.csv",
        OUTPUT_DATA / "profile_metrics_positive_8_full.csv",
        OUTPUT_DATA / "rater_metrics_full.csv",
        OUTPUT_DATA / "rating_distribution_full.csv",
        OUTPUT_FIGURES / "rating_distribution_full.png",
        OUTPUT_FIGURES / "strength_ccdf_full.png",
        OUTPUT_FIGURES / "popularity_vs_prestige_2_full.png",
        OUTPUT_FIGURES / "profile_lorenz_full.png",
    ]
    if force or any(not path.exists() for path in stage_two):
        run_script("02_full_project_analysis.py")

    stage_three = [
        OUTPUT_DATA / "profile_rating_extremes_summary_full.csv",
        OUTPUT_DATA / "profile_feature_alignment_overlap_full.csv",
        OUTPUT_DATA / "profile_feature_alignment_feature_classes_full.csv",
        OUTPUT_FIGURES / "profile_feature_alignment_consistency_full.png",
    ]
    if force or any(not path.exists() for path in stage_three):
        run_script("03_profile_rating_extremes.py")
        run_script("04_profile_feature_alignment.py")

    degree_fit = [
        OUTPUT_DATA / "degree_distribution_fit_full.json",
        OUTPUT_FIGURES / "degree_distribution_fit_full.png",
    ]
    if force or any(not path.exists() for path in degree_fit):
        run_script("05_degree_distribution_fit.py")


ensure_all_outputs(FORCE_REBUILD)

profile_metrics = pd.read_csv(OUTPUT_DATA / "profile_metrics_full.csv")
positive_profile_metrics = pd.read_csv(OUTPUT_DATA / "profile_metrics_positive_8_full.csv")
degree_fit = json.loads((OUTPUT_DATA / "degree_distribution_fit_full.json").read_text(encoding="utf-8"))
concentration_summary = pd.read_csv(OUTPUT_DATA / "profile_rating_extremes_summary_full.csv")
concentration_buckets = pd.read_csv(OUTPUT_DATA / "profile_rating_extremes_buckets_full.csv")
alignment_overlap = pd.read_csv(OUTPUT_DATA / "profile_feature_alignment_overlap_full.csv")
alignment_feature_classes = pd.read_csv(OUTPUT_DATA / "profile_feature_alignment_feature_classes_full.csv")


def top_overlap(frame: pd.DataFrame, col_a: str, col_b: str, k: int = 100) -> dict[str, float]:
    a = set(frame.sort_values(col_a, ascending=False).head(k)["profile_id"].tolist())
    b = set(frame.sort_values(col_b, ascending=False).head(k)["profile_id"].tolist())
    inter = len(a & b)
    union = len(a | b)
    return {"k": k, "intersection": inter, "jaccard": (inter / union) if union else 0.0}


full_corr = {
    "pearson": float(profile_metrics["in_strength"].corr(profile_metrics["authority_score"], method="pearson")),
    "spearman": float(profile_metrics["in_strength"].corr(profile_metrics["authority_score"], method="spearman")),
}
pos_corr = {
    "pearson": float(positive_profile_metrics["in_strength"].corr(positive_profile_metrics["authority_score"], method="pearson")),
    "spearman": float(positive_profile_metrics["in_strength"].corr(positive_profile_metrics["authority_score"], method="spearman")),
}
overlap = top_overlap(profile_metrics, "in_strength", "authority_score", k=100)
gini_in_strength = PopularityPrestigeAnalyzer.gini(profile_metrics["in_strength"])
gini_authority = PopularityPrestigeAnalyzer.gini(profile_metrics["authority_score"])

bucket_order = ["Top 1%", "Top 1-5%", "Top 5-10%", "Top 10-20%", "Top 20-50%", "Bottom 50%"]
bucket_view = concentration_buckets.pivot(index="bucket", columns="series", values="value_share").reindex(bucket_order)
top1_overlap = alignment_overlap.loc[alignment_overlap["bucket"] == "Top 1%"].iloc[0]


## Paper Figure 1 | Popularity and Prestige Alignment


In [ ]:
display(
    Markdown(
        """
        This figure gives the clearest summary of the popularity-prestige relationship.
        """
    )
)
display(Image(filename=str(OUTPUT_FIGURES / "popularity_vs_prestige_2_full.png")))


## Paper Figures 2–4 | Concentration and Degree Distribution


In [ ]:
concentration_gallery = [
    ("strength_ccdf_full.png", "Heavy-tailed interaction on the profile and rater sides."),
    (
        "degree_distribution_fit_full.png",
        f"Profile in-degree with a fitted power-law tail: alpha = {degree_fit['power_law_alpha']:.2f}, x_min = {int(degree_fit['power_law_xmin'])}.",
    ),
    ("profile_lorenz_full.png", "Lorenz curves for in-strength and authority on the profile side."),
]

for filename, note in concentration_gallery:
    display(Markdown(f"### {filename}\n\n{note}"))
    display(Image(filename=str(OUTPUT_FIGURES / filename)))


## Paper Figure 5 | Feature Alignment


In [ ]:
display(
    Markdown(
        """
        This figure links the popularity-prestige results with the concentration analysis.
        """
    )
)
display(Image(filename=str(OUTPUT_FIGURES / "profile_feature_alignment_consistency_full.png")))


## Reference Values

The table below collects the values that the paper text should agree with.


In [ ]:
reference_values = pd.DataFrame(
    [
        {"section": "Popularity vs prestige", "metric": "Pearson correlation", "value": full_corr["pearson"]},
        {"section": "Popularity vs prestige", "metric": "Spearman correlation", "value": full_corr["spearman"]},
        {"section": "Popularity vs prestige", "metric": "Top-100 overlap", "value": overlap["intersection"]},
        {"section": "Popularity vs prestige", "metric": "Top-100 Jaccard", "value": overlap["jaccard"]},
        {"section": "Positive layer", "metric": "Pearson correlation", "value": pos_corr["pearson"]},
        {"section": "Positive layer", "metric": "Spearman correlation", "value": pos_corr["spearman"]},
        {"section": "Concentration", "metric": "Gini of in-strength", "value": gini_in_strength},
        {"section": "Concentration", "metric": "Gini of authority", "value": gini_authority},
        {"section": "Concentration", "metric": "Top 1% share of all interactions", "value": bucket_view.loc["Top 1%", "all_interactions"]},
        {"section": "Concentration", "metric": "Top 1% share of high ratings", "value": bucket_view.loc["Top 1%", "high_ratings"]},
        {"section": "Concentration", "metric": "Top 1% share of low ratings", "value": bucket_view.loc["Top 1%", "low_ratings"]},
        {"section": "Feature alignment", "metric": "Top 1% overlap count", "value": top1_overlap["intersection_size"]},
        {"section": "Feature alignment", "metric": "Top 1% overlap Jaccard", "value": top1_overlap["jaccard_overlap"]},
        {"section": "Feature alignment", "metric": "Top 1% expected random overlap", "value": top1_overlap["expected_random_intersection"]},
        {"section": "Degree fit", "metric": "Power-law alpha", "value": degree_fit["power_law_alpha"]},
        {"section": "Degree fit", "metric": "Power-law x_min", "value": degree_fit["power_law_xmin"]},
        {"section": "Degree fit", "metric": "Nodes in tail", "value": degree_fit["n_in_tail"]},
        {"section": "Degree fit", "metric": "Power-law vs log-normal R", "value": degree_fit["vs_lognormal_R"]},
        {"section": "Degree fit", "metric": "Power-law vs exponential R", "value": degree_fit["vs_exponential_R"]},
    ]
)

display(reference_values)


In [ ]:
concentration_direction = "raw attention is slightly more concentrated than authority" if gini_in_strength > gini_authority else "authority is slightly more concentrated than raw attention"

display(
    Markdown(
        f"""
        ## Summary

        - The final figures support the same broad story as the analysis notebooks.
        - Popularity and prestige are strongly aligned, but not identical.
        - The profile side is extremely concentrated.
        - In the current outputs, **{concentration_direction}**.
        - The Top 1% overlap between interaction and high-rating buckets is **{top1_overlap['intersection_size']:,}** profiles, far above the random baseline of **{top1_overlap['expected_random_intersection']:.1f}**.
        """
    )
)


## Notes

This notebook does not edit the paper automatically.
It keeps the final figures and reference values in one place for checking.
